In [ ]:
import glob
import h5py
import importlib
import IPython.display as ipd
import soxr
import numpy as np
import os
import pandas as pd
import pickle
import soundfile as sf
import src.spatial_attn_lightning as binaural_lightning
import src.audio_transforms as at
importlib.reload(binaural_lightning)
import sys
import torch
import tqdm
import yaml

from pathlib import Path
from pytorch_lightning import Trainer
from scipy import signal
from scipy.io.wavfile import read, write

sys.path.append('../')
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

In [ ]:
h5_file = '/om2/user/rphess/Auditory-Attention/Room_HRIRs_50kHz.h5'
hrirs = h5py.File(h5_file, 'r', swmr=True)

In [ ]:
speaker_dir = '/om2/user/imgriff/datasets/commonvoice_9_en/3000ms/stimSR_50000/cv_9_en/subsets/'
speaker_paths = glob.glob(speaker_dir + 'train_core/*')
speaker_h5 = h5py.File(speaker_paths[0], 'r', swmr=True)

In [ ]:
def spatialize_sound(y, brir):
    """
    This takes a left-aligned BRIR and convolves it with a left-padded signal
    (using "valid" padding) to produce the same output as "same" padding with
    a center-aligned BRIR. It is faster to pad the signal than it is to pad
    the BRIR.
    
    Args
    ----
    y (np.ndarray): monoaural waveform with shape [timesteps]
    brir (np.ndarray): binaural room impulse response with shape [timesteps, 2]
    
    Returns
    -------
    y_spatialized (np.ndarray): binaural waveform with shape [timesteps, 2]
    """
    y_padded = np.pad(y, (brir.shape[0] - 1, 0))
    y_spatialized_l = signal.convolve(y_padded, brir[:, 0], mode='valid', method='direct')
    y_spatialized_r = signal.convolve(y_padded, brir[:, 1], mode='valid', method='direct')
    return np.stack([y_spatialized_l, y_spatialized_r]).T

Want to use code from above to generate validation/test sets to evaluate the model's performance with two speakers at the same location and 90 degrees apart

In [ ]:
rand_room = np.random.choice(list(hrirs.keys()))
room = hrirs[rand_room]
listener_locs = list(room.keys())
listener = room[listener_locs[0]]
impulse_response = listener['irs']

In [ ]:
ixs_0_elev = [ix for ix, elev in enumerate(listener['elevation']) if elev == 0]

# find five indexes that correspond to 5 azimuths that are 45 degrees apart out of the ixs_0_elev
ixs = []
for ix in ixs_0_elev:
    if listener['azimuth'][ix] in (0, 45, 90, 315, 270):
        ixs.append(ix)

In [ ]:
# picking two different speakers at random
speaker1_int = np.random.randint(0, len(speaker_h5['label_talker_int']))
speaker1_label = speaker_h5['label_talker_int'][speaker1_int]
same = True
while same:
    speaker2_int = np.random.randint(0, len(speaker_h5['label_talker_int']))
    if speaker_h5['label_talker_int'][speaker2_int] != speaker1_label:
        same = False

In [ ]:
# spatializing selected speakers
speaker1 = speaker_h5['signal'][speaker1_int]
speaker2 = speaker_h5['signal'][speaker2_int]
spatialized_samples = {'speaker1': dict(), 'speaker2': dict()}
for ix in ixs:
    azim = int(listener['azimuth'][ix])
    elev = int(listener['elevation'][ix])
    brir = impulse_response[ix]
    spatialized1 = spatialize_sound(speaker1, brir)
    spatialized2 = spatialize_sound(speaker2, brir)
    spatialized_samples['speaker1'][(azim, elev)] = spatialized1
    spatialized_samples['speaker2'][(azim, elev)] = spatialized2

Automate all code above to make multiple samples and test speed in a model

In [ ]:
# load in and randomly initialize model
config_path = "config/binaural_attn/word_task_both_cue.yml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

config['num_workers'] = 10
config['hparas']['batch_size'] = 64
config['audio']['rep_kwargs']['rep_on_gpu'] = True

In [ ]:
module = binaural_lightning.BinauralAttentionModule(config)

In [ ]:
# randomize values of model parameters
for p in module.parameters():
    p.data = torch.nn.parameter.Parameter(torch.randn_like(p))

In [ ]:
def spatialize_sound(y, brir):
    """
    This takes a left-aligned BRIR and convolves it with a left-padded signal
    (using "valid" padding) to produce the same output as "same" padding with
    a center-aligned BRIR. It is faster to pad the signal than it is to pad
    the BRIR.
    
    Args
    ----
    y (np.ndarray): monoaural waveform with shape [timesteps]
    brir (np.ndarray): binaural room impulse response with shape [timesteps, 2]
    
    Returns
    -------
    y_spatialized (np.ndarray): binaural waveform with shape [timesteps, 2]
    """
    y_padded = np.pad(y, (brir.shape[0] - 1, 0))
    y_spatialized_l = signal.convolve(y_padded, brir[:, 0], mode='valid', method='direct')
    y_spatialized_r = signal.convolve(y_padded, brir[:, 1], mode='valid', method='direct')
    return np.stack([y_spatialized_l, y_spatialized_r]).T

In [ ]:
h5_file = '/om2/user/rphess/Auditory-Attention/Room_HRIRs_50kHz.h5'
hrirs = h5py.File(h5_file, 'r', swmr=True)

rand_room = np.random.choice(list(hrirs.keys()))
room = hrirs[rand_room]
listener_locs = list(room.keys())
listener = room[listener_locs[0]]
impulse_response = listener['irs']

ixs_0_elev = [ix for ix, elev in enumerate(listener['elevation']) if elev == 0]

# find five indexes that correspond to 5 azimuths that are 45 degrees apart out of the ixs_0_elev
ixs = []
for ix in ixs_0_elev:
    if listener['azimuth'][ix] in (0, 90):
        ixs.append(ix)
ixs

In [ ]:
speaker_dir = '/om2/user/imgriff/datasets/commonvoice_9_en/3000ms/stimSR_50000/cv_9_en/subsets/'
speaker_paths = glob.glob(speaker_dir + 'train_core/*')
speaker_h5 = h5py.File(speaker_paths[0], 'r', swmr=True)

In [ ]:
speaker_h5.keys()

In [ ]:
# now generate 200 examples of spatialized speech with 2 speakers
for _ in tqdm.tqdm(range(200)):
    # picking two different speakers at random
    speaker1_int = np.random.randint(0, len(speaker_h5['label_talker_int']))
    speaker1_label = speaker_h5['label_talker_int'][speaker1_int]
    same = True
    while same:
        speaker2_int = np.random.randint(0, len(speaker_h5['label_talker_int']))
        if speaker_h5['label_talker_int'][speaker2_int] != speaker1_label:
            same = False

    # spatializing selected speakers
    speaker1 = speaker_h5['signal'][speaker1_int]
    speaker2 = speaker_h5['signal'][speaker2_int]
    spatialized_samples = {'speaker1': dict(), 'speaker2': dict()}
    for ix in ixs:
        azim = int(listener['azimuth'][ix])
        elev = int(listener['elevation'][ix])
        brir = impulse_response[ix]
        spatialized1 = spatialize_sound(speaker1, brir)
        spatialized2 = spatialize_sound(speaker2, brir)
        spatialized_samples['speaker1'][(azim, elev)] = spatialized1
        spatialized_samples['speaker2'][(azim, elev)] = spatialized2

    # grab cue and label then pass through model
    label = speaker_h5['label_word_int'][speaker1_int]
    fg = torch.from_numpy(spatialized_samples['speaker1'][(90, 0)].T).view(1, 2, -1)
    bg = torch.from_numpy(spatialized_samples['speaker2'][(0, 0)].T).view(1, 2, -1)
    module(fg, bg, None)

Creating an h5 with each word spatialized at each speaker location

In [ ]:
print("Loading speaker array room BRIRs")
list_data_dict = []
for elev in [-20, -10, 0, 10, 20, 30, 40]:
    for azim in np.arange(0, 360, 5):
        data_dict = {
            'azim': azim,
            'elev': elev,
            'brir': [],
        }
        for ear in ['l', 'r']:
            basename = f'{elev}elev_{azim}az_2.47x2.60y2.00z_{ear}.wav'
            if elev >= 0:
                fn = os.path.join('/om/user/francl/Room_Simulator_20181115_Rebuild/room_HRIRs/', basename)
            else:
                fn = os.path.join('/om/user/francl/Room_Simulator_20181115_Rebuild/room_HRIRs/neg_elevs/', basename)
            assert os.path.exists(fn)
            brir, sr_src = sf.read(fn)
            sr = 50000
            brir = soxr.resample(brir.astype(np.float32), sr_src, sr)
            data_dict['brir'].append(brir)
        data_dict['sr'] = sr
        data_dict['brir'] = np.stack(data_dict['brir']).T
        list_data_dict.append(data_dict)
df_brir = pd.DataFrame(list_data_dict)
df_brir_room = df_brir[np.logical_and.reduce([
    df_brir['azim'] % 10 == 0,
    ~(np.logical_and(df_brir['azim'] > 90, df_brir['azim'] < 270)),
    df_brir['elev'] >= 0,
])].reset_index()
print(f"Loaded speaker array room BRIRs ({len(df_brir_room)})")

In [ ]:
word_dir = '/om2/user/msaddler/spatial_audio_pipeline/assets/human_experiment_v00/foreground_swc/'
word_paths = glob.glob(word_dir + '*')
word_data = pickle.load(open(word_paths[-1], 'rb'))


In [ ]:
word_data

In [ ]:
df_brir_room

In [ ]:
all_data = [read(fn) for fn in word_data['src_fn']]

In [ ]:
all_words_resampled = [soxr.resample(data[1].astype(np.float32), data[0], 50000) for data in all_data]

In [ ]:
all_words_resampled = [torch.Tensor(data) for data in all_words_resampled]
all_words = torch.stack(all_words_resampled)

In [ ]:
irs = df_brir_room['brir'].values
irs = [torch.flip(torch.Tensor(ir), dims=[0]) for ir in irs]
irs = torch.stack(irs)

In [ ]:
spatial_test = torch.nn.functional.conv1d(all_words_resampled[5].view(5, 1, -1).cuda(), irs[0].view(2, 1, -1).cuda()).cuda()

In [ ]:
spatial_test.view(5, 2, -1).shape

In [ ]:
sounds = mass_spatialize(all_words[:5], irs[0])

In [ ]:
sounds.shape

In [ ]:
def mass_spatialize(words, ir):
    """Uses pytorch to convolve all sounds in words with 2 channel IR given in ir"""
    n_words = words.shape[0]
    words_padded = [torch.nn.functional.pad(word, (ir.shape[0] - 1, 0)) for word in words]
    ir = ir.T.unsqueeze(1)
    words_padded = torch.stack(words_padded)
    spatialized = torch.nn.functional.conv1d(words_padded.view(n_words, 1, -1).cuda(), ir.cuda()).cuda()
    return spatialized

In [ ]:
!nvidia-smi

In [ ]:
n_words = all_words.shape[0]
n_irs = irs.shape[0]
with h5py.File('first_376.h5', 'w') as h5:
    for i in tqdm.tqdm(range(n_irs)):
        azim = df_brir_room['azim'][i]
        elev = df_brir_room['elev'][i]
        brir = irs[i]
        loc_group = h5.create_group(f'{azim}_{elev}')
        talker_id_kernel = np.array(word_data['client_id'], dtype='object')
        word_kernel = np.array(word_data['word'], dtype='object')
        signal_kernel = mass_spatialize(all_words, brir)
        loc_group.create_dataset('talker_id', data=talker_id_kernel)
        loc_group.create_dataset('word', data=word_kernel)
        loc_group.create_dataset('signal', data=signal_kernel.cpu().numpy())

In [ ]:
h5 = h5py.File('first_376.h5', 'r')

In [ ]:
h5.keys()